In [9]:
import numpy as np
import pandas as pd
import seaborn as sns
import nltk
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import plotly.express as px
from random import randint
%matplotlib inline

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [11]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import one_hot,Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Embedding, Input, LSTM, Conv1D, MaxPool1D, Bidirectional, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

In [12]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [13]:
df = pd.read_csv("Data/stock_tweet_sentiment.csv")
df.head(3)

,Unnamed: 0,text,timestamp,source,symbols,company_names,Sentiment
0,0,VIDEO: “I was in my office. I was minding my o...,Wed Jul 18 21:33:26 +0000 2018,GoldmanSachs,GS,The Goldman Sachs,0
1,1,The price of lumber $LB_F is down 22% since hi...,Wed Jul 18 22:22:47 +0000 2018,StockTwits,M,Macy's,0
2,2,Who says the American Dream is dead? https://t...,Wed Jul 18 22:32:01 +0000 2018,TheStreet,AIG,American,-1


In [14]:
import string

def remove_punc(message):
    punc_removed = [char for char in message if char not in string.punctuation]
    punc_removed_join = ''.join(punc_removed)

    return punc_removed_join

In [15]:
if "text" in df.columns:
    df["text"] = df["text"].apply(remove_punc)

In [16]:
import gensim

nltk.download("stopwords")
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use','will','aap','co','day','user','stock','today','week','year','https', 'httpstco', 'httpstco'])

[nltk_data] Downloading package stopwords to
[nltk_data]     c:\Users\phari\.conda\envs\fyp\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if len(token) >= 3 and token not in stop_words:
            result.append(token)
            
    return result

In [18]:
if "text" in df.columns:
    df["text"] = df["text"].apply(preprocess)

In [19]:
if "text" in df.columns:
    df["text"] = df["text"].apply(lambda x: " ".join(x))

In [20]:
# Ensure sentiment labels are mapped before splitting
df["Sentiment"] = df["Sentiment"].map({-1: 0, 0: 1, 1: 2}).astype(int)

# Verify the mapping worked
print("Unique Labels After Mapping:", df["Sentiment"].unique())  # Should print [0 1 2]

Unique Labels After Mapping: [1 0 2]


In [21]:
# Assuming your CSV has columns "text" and "label"
texts = df['text'].tolist()
labels = df['Sentiment'].tolist()

In [22]:
# 2. Split data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [23]:
# 3. Initialize FinBERT Tokenizer
# You can use a FinBERT model available on Hugging Face, e.g., 'yiyanghkust/finbert-tone' or 'ProsusAI/finbert'
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')

In [24]:
# 4. Create a custom Dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        # Remove the extra batch dimension and prepare the item
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [25]:
# Create dataset objects
train_dataset = TweetDataset(train_texts, train_labels, tokenizer)
val_dataset = TweetDataset(val_texts, val_labels, tokenizer)

In [26]:
# 5. Load the pre-trained FinBERT model for sequence classification
# Set the num_labels parameter to match the number of sentiment classes (e.g., 3 for negative, neutral, positive)
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', num_labels=3)

In [27]:
# 6. Define training arguments and the Trainer
training_args = TrainingArguments(
    output_dir='./finbert_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",        # Save the model at the end of each epoch
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Define a simple compute_metrics function to evaluate accuracy and F1 score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

c:\Users\phari\.conda\envs\fyp\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# 7. Fine tune the model
#trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.172600,0.271657,0.944093,0.943474
2,0.040700,0.212566,0.961674,0.961542
3,0.002500,0.192270,0.968882,0.968719


TrainOutput(global_step=4266, training_loss=0.18962149976660353, metrics={'train_runtime': 2720.8042, 'train_samples_per_second': 25.087, 'train_steps_per_second': 1.568, 'total_flos': 4489767360110592.0, 'train_loss': 0.18962149976660353, 'epoch': 3.0})

In [ ]:
# 8. Save the fine tuned model and tokenizer
#model.save_pretrained('./finbert_finetuned')
#tokenizer.save_pretrained('./finbert_finetuned')

('./finbert_finetuned\\tokenizer_config.json',
 './finbert_finetuned\\special_tokens_map.json',
 './finbert_finetuned\\vocab.txt',
 './finbert_finetuned\\added_tokens.json')

In [30]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Load the saved model and tokenizer
model_path = "./finbert_finetuned"
loaded_model = BertForSequenceClassification.from_pretrained(model_path)
loaded_tokenizer = BertTokenizer.from_pretrained(model_path)

print("Model and tokenizer successfully loaded!")

Model and tokenizer successfully loaded!


In [31]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Move model to GPU if available
device = torch.device("cpu")
loaded_model.to(device)

# Function to predict batch-wise
def get_predictions(model, dataset):
    model.eval()  # Set model to evaluation mode
    all_preds, all_labels = [], []
    
    for batch in val_dataset:
        inputs = {
            "input_ids": batch["input_ids"].unsqueeze(0).to(device),
            "attention_mask": batch["attention_mask"].unsqueeze(0).to(device),
        }
        labels = batch["labels"].item()
        
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        
        all_preds.append(predictions[0])
        all_labels.append(labels)
    
    return np.array(all_preds), np.array(all_labels)

# Get predicted and true labels
predicted_labels, true_labels = get_predictions(loaded_model, val_dataset)

In [32]:
# Compute accuracy
accuracy = accuracy_score(true_labels, predicted_labels)

# Compute precision, recall, and F1-score
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average="weighted")

# Display results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.9689
Precision: 0.9689
Recall: 0.9689
F1 Score: 0.9687
